# V6: Cross-Run Long-Term Memory

V5 can pause, checkpoint, resume, replan, and finish a long-running task. V6 adds a different capability: **learning from one completed run and selectively reusing that knowledge in a later run**.

This notebook builds long-term memory from scratch in pure Python. It intentionally separates:

- **Conversation state**: messages in the current interaction.
- **Working memory**: compact state needed by the current run.
- **Checkpoint**: a durable snapshot used to resume the same run.
- **Long-term memory**: selected knowledge retrieved by future runs.

The key design principle is that storing everything is not memory design. A useful memory system needs write policy, provenance, retrieval, conflict handling, and forgetting.

## 0. Source Map and Design Rationale

This implementation is an educational synthesis of current first-party guidance and primary research:

- [OpenAI Conversation State](https://developers.openai.com/api/docs/guides/conversation-state): separates API conversation continuity from application-owned durable memory.
- [Google Cloud: Agent State and Memory with ADK](https://cloud.google.com/blog/topics/developers-practitioners/remember-this-agent-state-and-memory-with-adk): distinguishes session state from cross-session long-term memory and explains why external persistence matters.
- [Google Cloud Agentic Architecture Components](https://docs.cloud.google.com/architecture/choose-agentic-ai-architecture-components): recommends external state for scalable, stateless agent runtimes.
- [MemGPT](https://arxiv.org/abs/2310.08560): frames memory management as movement between limited active context and external storage.

V6 uses three memory classes:

1. **Episodic** — what happened in a particular run.
2. **Semantic** — stable facts or definitions learned from evidence.
3. **Procedural** — reusable strategies describing how to perform a task.

We store compact outcomes and strategies, not hidden chain-of-thought.

## 1. Setup

In [1]:
import json
import os
import re
from collections import Counter
from enum import Enum
from typing import Any

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(find_dotenv(usecwd=True))

MODEL_NAME = os.getenv("MODEL_NAME", "gpt-5-nano")
USE_LIVE_MODEL = os.getenv("V6_USE_LIVE_MODEL", "1") == "1"
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None

print(f"[V6] Model: {MODEL_NAME}")
print(f"[V6] Live synthesis enabled: {USE_LIVE_MODEL and client is not None}")

[V6] Model: gpt-5-nano
[V6] Live synthesis enabled: True


## 2. Memory Contracts

A memory record is not just text. Its metadata answers the questions that make memory safe to reuse:

- What kind of memory is it?
- Which run produced it?
- How confident are we?
- Is it still active, or has newer evidence superseded it?
- How often has it actually been retrieved?

`subject_key` gives semantic and procedural memories a stable identity so contradictory updates can be resolved explicitly.

In [2]:
class MemoryKind(str, Enum):
    EPISODIC = "episodic"
    SEMANTIC = "semantic"
    PROCEDURAL = "procedural"


class MemoryStatus(str, Enum):
    ACTIVE = "active"
    SUPERSEDED = "superseded"
    FORGOTTEN = "forgotten"


class MemoryWriteCandidate(BaseModel):
    kind: MemoryKind
    content: str = Field(min_length=12)
    tags: list[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    subject_key: str | None = None
    reusable: bool = True


class MemoryRecord(BaseModel):
    id: str
    kind: MemoryKind
    content: str
    tags: list[str]
    source_run_ids: list[str]
    confidence: float
    created_sequence: int
    last_access_sequence: int
    access_count: int = 0
    status: MemoryStatus = MemoryStatus.ACTIVE
    subject_key: str | None = None
    supersedes_id: str | None = None


class MemoryRetrieval(BaseModel):
    query: str
    records: list[MemoryRecord]
    scores: dict[str, float]


class MemoryPolicy(BaseModel):
    minimum_confidence: float = 0.65
    maximum_retrievals: int = 4
    forgetting_after_sequences: int = 8
    forgetting_confidence_ceiling: float = 0.70

## 3. Policy-Governed Long-Term Memory Store

The store implements five operations:

1. **Write gate** rejects transient or low-confidence content.
2. **Conflict resolution** keeps one active version per `subject_key`.
3. **Retrieval** combines token overlap, tag overlap, confidence, and use history.
4. **Forgetting** removes stale, low-confidence, unused memories.
5. **Serialization** proves that memory can survive a process boundary.

The scoring function is deliberately simple and inspectable. A production version could replace it with embeddings or hybrid retrieval without changing the contract.

In [3]:
def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))


class LongTermMemoryStore:
    def __init__(self, policy: MemoryPolicy | None = None):
        self.policy = policy or MemoryPolicy()
        self.records: dict[str, MemoryRecord] = {}
        self.sequence = 0
        self.rejections: list[dict[str, str]] = []

    def _active_subject(self, subject_key: str | None) -> MemoryRecord | None:
        if subject_key is None:
            return None
        return next(
            (
                record
                for record in self.records.values()
                if record.subject_key == subject_key
                and record.status == MemoryStatus.ACTIVE
            ),
            None,
        )

    def write(
        self,
        candidate: MemoryWriteCandidate,
        source_run_id: str,
    ) -> MemoryRecord | None:
        self.sequence += 1

        if not candidate.reusable:
            self.rejections.append(
                {"content": candidate.content, "reason": "not_reusable"}
            )
            return None

        if candidate.confidence < self.policy.minimum_confidence:
            self.rejections.append(
                {"content": candidate.content, "reason": "low_confidence"}
            )
            return None

        existing = self._active_subject(candidate.subject_key)
        if existing and candidate.confidence <= existing.confidence:
            self.rejections.append(
                {"content": candidate.content, "reason": "weaker_conflict"}
            )
            return None

        record_id = f"memory_{len(self.records) + 1}"
        record = MemoryRecord(
            id=record_id,
            kind=candidate.kind,
            content=candidate.content.strip(),
            tags=sorted(set(candidate.tags)),
            source_run_ids=[source_run_id],
            confidence=candidate.confidence,
            created_sequence=self.sequence,
            last_access_sequence=self.sequence,
            subject_key=candidate.subject_key,
            supersedes_id=existing.id if existing else None,
        )

        if existing:
            existing.status = MemoryStatus.SUPERSEDED

        self.records[record.id] = record
        return record

    def retrieve(
        self,
        query: str,
        kinds: set[MemoryKind] | None = None,
        top_k: int | None = None,
    ) -> MemoryRetrieval:
        self.sequence += 1
        query_tokens = tokenize(query)
        scored: list[tuple[float, MemoryRecord]] = []

        for record in self.records.values():
            if record.status != MemoryStatus.ACTIVE:
                continue
            if kinds and record.kind not in kinds:
                continue

            content_tokens = tokenize(record.content)
            tag_tokens = set(record.tags)
            overlap = len(query_tokens & content_tokens) / max(len(query_tokens), 1)
            tag_overlap = len(query_tokens & tag_tokens) / max(len(query_tokens), 1)
            reuse_bonus = min(record.access_count, 3) * 0.02
            score = overlap * 0.55 + tag_overlap * 0.20 + record.confidence * 0.25
            score += reuse_bonus
            if score > 0.15:
                scored.append((round(score, 4), record))

        limit = top_k or self.policy.maximum_retrievals
        selected = sorted(scored, key=lambda item: (-item[0], item[1].id))[:limit]
        for _, record in selected:
            record.access_count += 1
            record.last_access_sequence = self.sequence

        return MemoryRetrieval(
            query=query,
            records=[record.model_copy(deep=True) for _, record in selected],
            scores={record.id: score for score, record in selected},
        )

    def forget_stale(self) -> list[str]:
        self.sequence += 1
        forgotten: list[str] = []
        for record in self.records.values():
            age = self.sequence - record.last_access_sequence
            should_forget = (
                record.status == MemoryStatus.ACTIVE
                and record.access_count == 0
                and record.confidence <= self.policy.forgetting_confidence_ceiling
                and age >= self.policy.forgetting_after_sequences
            )
            if should_forget:
                record.status = MemoryStatus.FORGOTTEN
                forgotten.append(record.id)
        return forgotten

    def to_json(self) -> str:
        payload = {
            "policy": self.policy.model_dump(mode="json"),
            "sequence": self.sequence,
            "records": [record.model_dump(mode="json") for record in self.records.values()],
            "rejections": self.rejections,
        }
        return json.dumps(payload, indent=2)

    @classmethod
    def from_json(cls, payload: str) -> "LongTermMemoryStore":
        data = json.loads(payload)
        store = cls(MemoryPolicy.model_validate(data["policy"]))
        store.sequence = data["sequence"]
        store.records = {
            item["id"]: MemoryRecord.model_validate(item)
            for item in data["records"]
        }
        store.rejections = data["rejections"]
        return store

## 4. Simulated Research Environment

Memory quality cannot be evaluated by asking whether retrieval “looks reasonable.” We need a task-level comparison.

The simulator below exposes the same research task to two agents:

- **Baseline** starts from scratch and explores broadly.
- **Memory-assisted** retrieves a proven procedural strategy and a semantic definition from a prior run.

The environment is deterministic so any improvement comes from the memory policy, not model randomness. The optional live model is used only to synthesize the final explanation.

In [4]:
class ResearchTask(BaseModel):
    id: str
    question: str
    required_lanes: set[str] = Field(default_factory=lambda: {"academic", "official"})


class ResearchRunResult(BaseModel):
    run_id: str
    task_id: str
    retrieved_memory_ids: list[str]
    strategy: str
    tool_calls: list[str]
    evidence_lanes: set[str]
    quality_score: float


def run_research_task(
    task: ResearchTask,
    run_id: str,
    memories: list[MemoryRecord] | None = None,
) -> ResearchRunResult:
    memories = memories or []
    has_procedure = any(
        memory.kind == MemoryKind.PROCEDURAL
        and "academic-first" in memory.content.lower()
        for memory in memories
    )
    has_definition = any(
        memory.kind == MemoryKind.SEMANTIC
        and "goal-directed execution" in memory.content.lower()
        for memory in memories
    )

    if has_procedure:
        tool_calls = [
            "arxiv_search: long-horizon agent planning evaluation",
            "official_docs_search: agent state memory evaluation",
        ]
        evidence_lanes = {"academic", "official"}
        strategy = "academic-first triangulation with an evidence stopping rule"
        quality = 0.94 + (0.04 if has_definition else 0.0)
    else:
        tool_calls = [
            "web_search: long-horizon AI",
            "web_search: autonomous agent blogs",
            "arxiv_search: LLM agent memory",
            "official_docs_search: agent architecture",
        ]
        evidence_lanes = {"blog", "academic", "official"}
        strategy = "broad exploration followed by source cleanup"
        quality = 0.82

    return ResearchRunResult(
        run_id=run_id,
        task_id=task.id,
        retrieved_memory_ids=[memory.id for memory in memories],
        strategy=strategy,
        tool_calls=tool_calls,
        evidence_lanes=evidence_lanes,
        quality_score=round(quality, 2),
    )

## 5. Run 1: Extract Memories from Experience

The first run discovers that a broad query is wasteful. Its post-run reflection emits three compact candidates:

- an episodic outcome,
- a semantic clarification,
- a procedural search strategy.

A transient UI observation and a low-confidence contradiction are deliberately submitted to prove that the write gate rejects them.

In [5]:
memory_store = LongTermMemoryStore()

training_task = ResearchTask(
    id="task_training",
    question="Study reliable long-horizon research agents and their evaluation.",
)
training_run = run_research_task(training_task, "run_001")

candidates = [
    MemoryWriteCandidate(
        kind=MemoryKind.EPISODIC,
        content="Run 001 used four calls because its first two broad web queries produced redundant evidence.",
        tags=["research", "tool-calls", "redundancy"],
        confidence=0.84,
        subject_key="run-001-search-outcome",
    ),
    MemoryWriteCandidate(
        kind=MemoryKind.SEMANTIC,
        content="Long-horizon agency means goal-directed execution across many decisions and state transitions, not merely a large context window.",
        tags=["long-horizon", "agent", "definition"],
        confidence=0.92,
        subject_key="long-horizon-definition",
    ),
    MemoryWriteCandidate(
        kind=MemoryKind.PROCEDURAL,
        content="For technical research, use an academic-first query, triangulate with official documentation, deduplicate sources, and stop when required evidence lanes are covered.",
        tags=["research", "academic-first", "evidence", "agent"],
        confidence=0.90,
        subject_key="technical-research-strategy",
    ),
]

stored_memories = [
    memory_store.write(candidate, source_run_id=training_run.run_id)
    for candidate in candidates
]

transient_memory = memory_store.write(
    MemoryWriteCandidate(
        kind=MemoryKind.EPISODIC,
        content="The notebook output panel was scrolled halfway down.",
        tags=["ui"],
        confidence=0.99,
        reusable=False,
    ),
    source_run_id=training_run.run_id,
)

weak_conflict = memory_store.write(
    MemoryWriteCandidate(
        kind=MemoryKind.SEMANTIC,
        content="Long-horizon agency is exactly the same thing as a large context window.",
        tags=["long-horizon", "definition"],
        confidence=0.70,
        subject_key="long-horizon-definition",
    ),
    source_run_id="run_002_untrusted",
)

print(f"[V6][RUN_001] Stored memories: {sum(item is not None for item in stored_memories)}")
print(f"[V6][RUN_001] Rejected candidates: {len(memory_store.rejections)}")
print(f"[V6][RUN_001] Active kinds: {sorted(record.kind.value for record in memory_store.records.values())}")

[V6][RUN_001] Stored memories: 3
[V6][RUN_001] Rejected candidates: 2
[V6][RUN_001] Active kinds: ['episodic', 'procedural', 'semantic']


## 6. Run 2: Baseline vs Memory-Assisted Research

A new run serializes and restores the store before retrieval. This simulates a new process rather than accidentally reusing Python objects from Run 1.

Retrieval is scoped to semantic and procedural memory. The episode remains auditable but is not injected because the next task needs reusable knowledge, not the full story of the previous run.

In [6]:
serialized_store = memory_store.to_json()
restored_store = LongTermMemoryStore.from_json(serialized_store)

evaluation_task = ResearchTask(
    id="task_evaluation",
    question="Compare long-horizon agent memory and evaluation patterns using academic and official evidence.",
)

baseline_result = run_research_task(
    evaluation_task,
    run_id="run_002_baseline",
    memories=[],
)

retrieval = restored_store.retrieve(
    evaluation_task.question,
    kinds={MemoryKind.SEMANTIC, MemoryKind.PROCEDURAL},
    top_k=3,
)
assisted_result = run_research_task(
    evaluation_task,
    run_id="run_002_memory_assisted",
    memories=retrieval.records,
)

comparison = {
    "baseline": baseline_result.model_dump(mode="json"),
    "memory_assisted": assisted_result.model_dump(mode="json"),
    "retrieval_scores": retrieval.scores,
    "persisted_record_count": len(restored_store.records),
}

print("[V6][A/B] Baseline tool calls:", len(baseline_result.tool_calls))
print("[V6][A/B] Assisted tool calls:", len(assisted_result.tool_calls))
print("[V6][A/B] Baseline quality:", baseline_result.quality_score)
print("[V6][A/B] Assisted quality:", assisted_result.quality_score)
print("[V6][A/B] Retrieved memories:", assisted_result.retrieved_memory_ids)

[V6][A/B] Baseline tool calls: 4
[V6][A/B] Assisted tool calls: 2
[V6][A/B] Baseline quality: 0.82
[V6][A/B] Assisted quality: 0.98
[V6][A/B] Retrieved memories: ['memory_3', 'memory_2']


## 7. Conflict Update and Forgetting

Memory must be revisable. A stronger evidence-backed definition supersedes the prior semantic record rather than silently appending a contradiction.

We also add an intentionally marginal, never-retrieved episode and advance the logical sequence. The forgetting policy removes only stale, low-confidence, unused content; high-confidence and frequently retrieved memories remain active.

In [7]:
stronger_definition = restored_store.write(
    MemoryWriteCandidate(
        kind=MemoryKind.SEMANTIC,
        content="Long-horizon agency is goal-directed execution across many decisions, environment observations, and resumable state transitions; context length is only one supporting resource.",
        tags=["long-horizon", "agent", "definition", "checkpoint"],
        confidence=0.97,
        subject_key="long-horizon-definition",
    ),
    source_run_id="run_003_verified",
)

stale_episode = restored_store.write(
    MemoryWriteCandidate(
        kind=MemoryKind.EPISODIC,
        content="One exploratory query returned a mildly relevant community discussion.",
        tags=["exploration"],
        confidence=0.66,
        subject_key="marginal-query-outcome",
    ),
    source_run_id="run_003_verified",
)

for _ in range(restored_store.policy.forgetting_after_sequences):
    restored_store.sequence += 1
forgotten_ids = restored_store.forget_stale()

definition_versions = [
    (record.id, record.status.value, record.confidence)
    for record in restored_store.records.values()
    if record.subject_key == "long-horizon-definition"
]

print("[V6][CONFLICT] Definition versions:", definition_versions)
print("[V6][FORGETTING] Forgotten memories:", forgotten_ids)

[V6][CONFLICT] Definition versions: [('memory_2', 'superseded', 0.92), ('memory_4', 'active', 0.97)]
[V6][FORGETTING] Forgotten memories: ['memory_5']


## 8. End-to-End Evaluation

The evaluation checks behavior, not just serialization:

- all three memory types were created with provenance;
- low-value and weak contradictory candidates were rejected;
- a future run retrieved knowledge from a different run;
- the assisted run used fewer calls without reducing quality;
- stronger evidence superseded older semantic memory;
- forgetting affected only an eligible stale memory;
- persistence survived a round trip.

In [8]:
active_definition = next(
    record
    for record in restored_store.records.values()
    if record.subject_key == "long-horizon-definition"
    and record.status == MemoryStatus.ACTIVE
)

checks = {
    "three_memory_types_created": {
        record.kind for record in memory_store.records.values()
    } == set(MemoryKind),
    "all_memories_have_provenance": all(
        record.source_run_ids for record in memory_store.records.values()
    ),
    "transient_candidate_rejected": transient_memory is None,
    "weak_conflict_rejected": weak_conflict is None,
    "cross_run_memory_retrieved": bool(assisted_result.retrieved_memory_ids),
    "retrieval_originates_from_prior_run": all(
        "run_001" in record.source_run_ids for record in retrieval.records
    ),
    "assisted_uses_fewer_calls": len(assisted_result.tool_calls)
    < len(baseline_result.tool_calls),
    "assisted_preserves_quality": assisted_result.quality_score
    >= baseline_result.quality_score,
    "required_evidence_covered": evaluation_task.required_lanes
    <= assisted_result.evidence_lanes,
    "stronger_definition_is_active": active_definition.id
    == stronger_definition.id,
    "older_definition_superseded": any(
        status == MemoryStatus.SUPERSEDED.value
        for _, status, _ in definition_versions
    ),
    "stale_low_confidence_memory_forgotten": stale_episode.id in forgotten_ids,
    "serialization_round_trip": len(memory_store.records)
    == len(LongTermMemoryStore.from_json(serialized_store).records),
}

for name, passed in checks.items():
    print(f"[V6][EVAL] {name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values()), [name for name, passed in checks.items() if not passed]
print(f"[V6][EVAL] Passed {sum(checks.values())}/{len(checks)} checks.")

[V6][EVAL] three_memory_types_created: PASS
[V6][EVAL] all_memories_have_provenance: PASS
[V6][EVAL] transient_candidate_rejected: PASS
[V6][EVAL] weak_conflict_rejected: PASS
[V6][EVAL] cross_run_memory_retrieved: PASS
[V6][EVAL] retrieval_originates_from_prior_run: PASS
[V6][EVAL] assisted_uses_fewer_calls: PASS
[V6][EVAL] assisted_preserves_quality: PASS
[V6][EVAL] required_evidence_covered: PASS
[V6][EVAL] stronger_definition_is_active: PASS
[V6][EVAL] older_definition_superseded: PASS
[V6][EVAL] stale_low_confidence_memory_forgotten: PASS
[V6][EVAL] serialization_round_trip: PASS
[V6][EVAL] Passed 13/13 checks.


## 9. Embedded Memory-Aware Report

In [9]:
report_payload = {
    "experiment": comparison,
    "active_memory_count": sum(
        record.status == MemoryStatus.ACTIVE
        for record in restored_store.records.values()
    ),
    "superseded_memory_count": sum(
        record.status == MemoryStatus.SUPERSEDED
        for record in restored_store.records.values()
    ),
    "forgotten_memory_count": sum(
        record.status == MemoryStatus.FORGOTTEN
        for record in restored_store.records.values()
    ),
    "checks": checks,
}

fallback_report = (
    "# V6 Long-Term Memory Evaluation\n\n"
    "## Outcome\n\n"
    f"The cross-run memory policy passed {sum(checks.values())}/{len(checks)} checks. "
    f"The memory-assisted run reduced tool calls from {len(baseline_result.tool_calls)} "
    f"to {len(assisted_result.tool_calls)} while increasing the deterministic quality "
    f"score from {baseline_result.quality_score:.2f} to {assisted_result.quality_score:.2f}.\n\n"
    "## What Changed\n\n"
    "The second run retrieved a procedural academic-first search strategy and a semantic "
    "definition produced by an earlier run. It did not load the entire prior transcript. "
    "Provenance, conflict resolution, forgetting, and persistence remained explicit and "
    "inspectable.\n\n"
    "## Interpretation\n\n"
    "V6 demonstrates long-term memory as a governed cross-run subsystem. It is distinct "
    "from conversation history, V5 working memory, and V5 checkpoints. The improvement "
    "comes from selective reuse under evaluation, not from increasing context size."
)

live_report_used = False
final_report = fallback_report

if USE_LIVE_MODEL and client is not None:
    try:
        response = client.responses.create(
            model=MODEL_NAME,
            instructions=(
                "Write a concise Markdown engineering report from the supplied V6 evaluation JSON. "
                "Explain the distinction between conversation state, working memory, checkpoints, "
                "and cross-run long-term memory. Discuss measured results, safeguards, and limitations. "
                "Output only the report. Do not offer follow-up help."
            ),
            input=json.dumps(report_payload, indent=2),
        )
        if response.output_text.strip():
            final_report = response.output_text.strip()
            live_report_used = True
    except Exception as exc:
        print(f"[V6][REPORT] Live synthesis unavailable; using deterministic report: {type(exc).__name__}")

print(f"[V6][REPORT] Live model used: {live_report_used}")
print(f"[V6][REPORT] Characters: {len(final_report)}")
display(Markdown(final_report))

[V6][REPORT] Live model used: True
[V6][REPORT] Characters: 7309


# V6 Evaluation Engineering Report

Executive summary
- Memory-assisted evaluation shows a substantial quality uplift (0.98) over baseline (0.82).
- Cross-run long-term memory was leveraged, with 3 active memories retained across runs, 1 superseded, and 1 forgotten.
- Retrieved memories contributed modest per-item scores (memory_3: 0.4417; memory_2: 0.3842) but collectively supported higher-quality reasoning and evidence coverage.
- All implemented safeguards were reported as active, including provenance, evidence coverage, conflict rejection, and cross-run retrieval controls.
- Assessed efficiency improved: assisted approach used fewer calls while preserving quality.

Experiment overview
- Baseline (run_002_baseline)
  - Task: task_evaluation
  - Retrieved memory IDs: none
  - Strategy: broad exploration followed by source cleanup
  - Tool calls: web_search, arxiv_search, official_docs_search
  - Evidence lanes: blog, academic, official
  - Quality score: 0.82
- Memory-assisted (run_002_memory_assisted)
  - Task: task_evaluation
  - Retrieved memory IDs: memory_3, memory_2
  - Strategy: academic-first triangulation with an evidence stopping rule
  - Tool calls: arxiv_search, official_docs_search
  - Evidence lanes: academic, official
  - Quality score: 0.98
- Memory signals
  - Retrieval scores: memory_3 = 0.4417; memory_2 = 0.3842
  - Persisted record count across runs: 3
  - Active memory count (current run): 3
  - Superseded memory count: 1
  - Forgotten memory count: 1
- Safeguards and provenance
  - Checks: all listed safeguards returned true (see below)

Memory architecture concepts and distinctions

- Conversation state
  - Definition: the ephemeral context and reasoning from the current interaction or turn sequence.
  - Scope: exists within a single exchange or short session; not guaranteed to persist beyond the immediate succession of prompts and responses unless explicitly stored.

- Working memory
  - Definition: the in-session set of memories and evidentiary anchors actively used to reason, answer, and decide within a task run.
  - In the evaluation: active_memory_count = 3; retrieved_memory_ids indicate externally sourced elements incorporated into current reasoning.
  - Function: supports triangulation, evidence synthesis, and constraint satisfaction without committing to long-term persistence.

- Checkpoints
  - Definition: explicit, verifiable milestones for memory creation, provenance, and serialization that ensure integrity across runs.
  - In the evaluation: keys include three_memory_types_created, all_memories_have_provenance, transient_candidate_rejected, weak_conflict_rejected, required_evidence_covered, stronger_definition_is_active, older_definition_superseded, serialization_round_trip.
  - Purpose: prevent drift, preserve traceability, and ensure reproducibility and safety of memory handling.

- Cross-run long-term memory
  - Definition: memories and their provenance retained beyond a single run and retrievable by future sessions.
  - In the evaluation: cross_run_memory_retrieved = true; retrieval_originates_from_prior_run = true; memory IDs memory_2 and memory_3 were retrieved from prior run(s); persisted_record_count = 3.
  - Purpose: enable continuity, reuse of validated evidence, and cumulative knowledge across sessions.

Measured results and interpretation

- Quality and performance
  - Baseline quality score: 0.82
  - Memory-assisted quality score: 0.98
  - Interpretation: memory-assisted strategy substantially improves answer quality and justification through targeted evidence integration and better provenance management.

- Memory retrieval and impact
  - Retrieved memories: memory_2, memory_3
  - Individual retrieval scores: memory_3 = 0.4417; memory_2 = 0.3842
  - Interpretation: memory items contribute qualitative value despite modest per-item scores; their combination with an academic-first triangulation and evidence stopping rule enhances overall decision quality and defensibility.

- Memory economy and persistence
  - Persisted records: 3
  - Active memories: 3
  - Superseded memories: 1
  - Forgotten memories: 1
  - Interpretation: deliberate curation maintains a compact, relevant long-term memory set; stale or conflicting items are superseded or forgotten as appropriate.

- Safeguards and provenance checks
  - All checks reported as true:
    - three_memory_types_created
    - all_memories_have_provenance
    - transient_candidate_rejected
    - weak_conflict_rejected
    - cross_run_memory_retrieved
    - retrieval_originates_from_prior_run
    - assisted_uses_fewer_calls
    - assisted_preserves_quality
    - required_evidence_covered
    - stronger_definition_is_active
    - older_definition_superseded
    - stale_low_confidence_memory_forgotten
    - serialization_round_trip
  - Interpretation: robust governance over memory creation, provenance, conflict handling, evidence coverage, and cross-run consistency; serialization fidelity is confirmed.

Safeguards, risks, and limitations

- Safeguards
  - Provenance tracking for all memories
  - Cross-run retrieval with explicit origin tracing
  - Evidence coverage constraints and stopping rules
  - Conflict rejection and memory aging (stale/low-confidence memory handling)
  - Serialization and round-trip integrity checks
  - Outcome: strong defense-in-depth against memory drift, duplication, or contamination of reasoning.

- Limitations and caveats
  - Retrieval scores for individual memories remain moderate, indicating that not all retrieved items carry high intrinsic confidence; effective integration relies on higher-level synthesis and the evidence framework.
  - Cross-run memory relies on persistence mechanisms that must be maintained to prevent drift; long-term reliability depends on storage integrity and lineage history.
  - The evaluation focuses on a specific task type (task_evaluation) and a particular evidence framework; generalization to broader tasks should be validated.
  - Potential biases may arise from tool availability and evidence lanes (academic/official) and from the selection of triangulation strategies.

Conclusion

- The memory-assisted configuration demonstrates a pronounced improvement in evaluation quality by leveraging cross-run long-term memory with a strict provenance and evidence framework.
- Working memory remains a focused, per-run repository of relevant memories, while conversation state carries the ephemeral context needed for each turn.
- Checkpoints enforce integrity, serialization fidelity, and governance, enabling reliable cross-run memory reuse without compromising safety.
- The combination of these memory layers, under robust safeguards, yields higher-quality results and greater efficiency, albeit with recognition of moderate retrieval signals and task-bound limitations.

Notes on terminology mapping (to aid interpretation)
- Conversation state ≈ current-turn context and reasoning in the present interaction.
- Working memory ≈ in-session, actively used memories and evidence anchors (active_memory_count = 3).
- Checkpoints ≈ provenance, validation, and serialization milestones ensuring memory integrity.
- Cross-run long-term memory ≈ memories persisted and retrievable across runs (cross_run_memory_retrieved = true; persisted_record_count = 3).

## 10. V6 Completion Summary

V6 adds a complete educational memory lifecycle:

`experience -> candidate extraction -> write gate -> durable store -> retrieval -> task adaptation -> conflict update -> forgetting -> evaluation`

What V6 intentionally does **not** add is autonomous behavior modification. A procedural memory is still only retrieved context. V7 will turn proven procedures into versioned Skills, but only after evaluation and an explicit promotion decision.